# 🌽 Estimativa Local de Área Plantada de Milho (Google Colab)

Este notebook executa o pipeline de mapeamento de lavouras de milho usando imagens **Sentinel-2 L2A** do **Microsoft Planetary Computer**, análise fenológica temporal e Machine Learning diretamente na nuvem do Google.

Repositório oficial: **[github.com/wmkreuz/mapeamento_milho](https://github.com/wmkreuz/mapeamento_milho)**

---

### 💡 Vantagens de rodar no Google Colab:
- **Streaming Remoto de Imagens**: O Linux faz leitura remota de blocos (Cloud-Optimized GeoTIFFs), sem precisar baixar arquivos de 1 GB de cada cena.
- **Economia de Recursos Locais**: Não consome a memória RAM, CPU ou disco do seu computador.
- **Rede de Alta Velocidade**: O download e requisições STAC rodam diretamente nos servidores do Google.


### 📁 Passo 1: Conectar ao Google Drive (Recomendado)
Conectar ao Google Drive permite salvar os resultados de forma permanente, mesmo após fechar a sessão do Colab.


In [1]:
# @title 📁 Conectar ao Google Drive
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    print("✅ Google Drive montado com sucesso!")
except Exception as e:
    print("ℹ️ Google Drive não conectado. Os arquivos serão gerados no disco temporário do Colab.")


Mounted at /content/drive
✅ Google Drive montado com sucesso!


### 📂 Passo 2: Obter o Código do Repositório GitHub
Clona automaticamente o repositório **`wmkreuz/mapeamento_milho`** do GitHub.


In [2]:
# @title 📂 Clonar ou Atualizar o Repositório do GitHub
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wmkreuz/mapeamento_milho.git"
REPO_DIR = Path("/content/mapeamento_milho")

if not REPO_DIR.exists():
    print(f"📥 Clonando repositório de {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("🔄 Repositório já presente. Atualizando com git pull...")
    !git -C /content/mapeamento_milho pull

# Localiza onde está o main.py
possiveis_pastas = [
    REPO_DIR / "estimativa_milho",
    REPO_DIR,
    Path("/content/estimativa_milho"),
    Path("/content/drive/MyDrive/mapeamento_milho/estimativa_milho"),
]

pasta_projeto = None
for p in possiveis_pastas:
    if (p / "main.py").exists() and (p / "src").exists():
        pasta_projeto = p
        break

if pasta_projeto:
    os.chdir(pasta_projeto)
    print(f"✅ Diretório ativo configurado para: {pasta_projeto}")
else:
    raise FileNotFoundError("Não foi possível localizar 'main.py' no repositório clonado.")


📥 Clonando repositório de https://github.com/wmkreuz/mapeamento_milho.git...
Cloning into 'mapeamento_milho'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 33 (delta 1), reused 33 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 29.77 KiB | 1.24 MiB/s, done.
Resolving deltas: 100% (1/1), done.
✅ Diretório ativo configurado para: /content/mapeamento_milho/estimativa_milho


### ⚙️ Passo 3: Instalar as Dependências
Instala as bibliotecas geoespaciais e de machine learning necessárias (duração aproximada de 1 a 2 minutos).


In [3]:
# @title ⚙️ Instalar Dependências do Projeto
print("📦 Instalando dependências geoespaciais e de machine learning...")
!pip install -q --upgrade pip
!pip install -q geopandas rasterio rioxarray xarray dask scikit-learn scikit-image matplotlib folium pystac-client planetary-computer stackstac odc-stac geobr pyyaml jinja2 mapclassify

print("✅ Dependências instaladas com sucesso!")


📦 Instalando dependências geoespaciais e de machine learning...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.3 MB/s eta 0:00:00
✅ Dependências instaladas com sucesso!


### 🌾 Passo 4: Configurar os Parâmetros da Estimativa
Ajuste os parâmetros abaixo usando os campos interativos do formulário:


In [4]:
# @title 🌾 Configuração dos Parâmetros
# @markdown **Localização e Período:**
MUNICIPIO = "Braco do Norte"  # @param {type:"string"}
UF = "SC"  # @param {type:"string"}
SAFRA = "verao"  # @param ["verao", "safrinha", "manual"]
DATA_INICIO = "2023-09-01"  # @param {type:"date"}
DATA_FIM = "2024-04-30"  # @param {type:"date"}

# @markdown ---
# @markdown **Filtros e Limiares:**
MAX_CENAS = 12  # @param {type:"slider", min:4, max:30, step:1}
LIMIAR_PROVAVEL = 0.70  # @param {type:"slider", min:0.50, max:0.95, step:0.01}
LIMIAR_POSSIVEL = 0.50  # @param {type:"slider", min:0.30, max:0.80, step:0.01}
AREA_MINIMA_HA = 0.50  # @param {type:"number"}

# @markdown ---
# @markdown **Dados de Validação (Opcional):**
AREA_REFERENCIA_HA = 0.0  # @param {type:"number"}
CAMINHO_AMOSTRAS_MILHO = ""  # @param {type:"string"}
TREINO_REGIONAL = False  # @param {type:"boolean"}

# @markdown ---
# @markdown **Destino dos Resultados:**
SALVAR_NO_DRIVE = True  # @param {type:"boolean"}
PASTA_DESTINO_DRIVE = "/content/drive/MyDrive/estimativa_milho_saidas"  # @param {type:"string"}

import os
from pathlib import Path

if SALVAR_NO_DRIVE and os.path.exists("/content/drive/MyDrive"):
    pasta_saida = Path(PASTA_DESTINO_DRIVE)
else:
    pasta_saida = Path("data/output")

pasta_saida.mkdir(parents=True, exist_ok=True)
print(f"📁 Pasta de saída definida: {pasta_saida.resolve()}")


📁 Pasta de saída definida: /content/drive/MyDrive/estimativa_milho_saidas


### 🚀 Passo 5: Executar o Processamento
Busca as cenas Sentinel-2, mascara nuvens, calcula índices espectrais, aplica regras fenológicas e Random Forest.


In [5]:
# @title 🚀 Executar Estimativa
import subprocess
import sys
from pathlib import Path

comando = [
    sys.executable, "main.py",
    "--municipio", MUNICIPIO,
    "--uf", UF,
    "--data-inicio", str(DATA_INICIO),
    "--data-fim", str(DATA_FIM),
    "--safra", SAFRA,
    "--max-cenas", str(MAX_CENAS),
    "--limiar-provavel", str(LIMIAR_PROVAVEL),
    "--limiar-possivel", str(LIMIAR_POSSIVEL),
    "--area-minima-ha", str(AREA_MINIMA_HA),
    "--pasta-saida", str(pasta_saida)
]

if AREA_REFERENCIA_HA > 0:
    comando.extend(["--area-referencia-ha", str(AREA_REFERENCIA_HA)])

if CAMINHO_AMOSTRAS_MILHO and Path(CAMINHO_AMOSTRAS_MILHO).exists():
    comando.extend(["--amostras-milho", CAMINHO_AMOSTRAS_MILHO])
    if TREINO_REGIONAL:
        comando.append("--treino-regional")

print("▶️ Executando pipeline...")
print("Comando:", " ".join(comando))
print("-" * 60)

processo = subprocess.run(comando, text=True)

if processo.returncode == 0:
    print("🎉 Processamento concluído com sucesso!")
else:
    print("❌ Ocorreu um erro durante a execução. Verifique o log acima.")


SyntaxError: unterminated string literal (detected at line 35) (1355485440.py, line 35)

### 📊 Passo 6: Visualização dos Resultados
Exibe a tabela com hectares por classe, o gráfico da curva fenológica e o mapa interativo Folium diretamente no notebook.


In [ ]:
# @title 📊 Visualizar Resumo, Gráfico e Mapa Interativo
import os
import pandas as pd
from IPython.display import display, Image, HTML
from src.utils import slugify

slug = slugify(f"{MUNICIPIO}_{UF}")
pasta_resultado = pasta_saida / slug

resumo_csv = pasta_resultado / "resumo.csv"
grafico_png = pasta_resultado / "grafico_serie_temporal.png"
mapa_html = pasta_resultado / "mapa_interativo.html"
relatorio_html = pasta_resultado / "relatorio_estimativa.html"

# 1. Resumo em Tabela
if resumo_csv.exists():
    print("📈 Resumo da Área Estimada:")
    df_resumo = pd.read_csv(resumo_csv)
    display(df_resumo)
else:
    print("⚠️ Arquivo resumo.csv não encontrado.")

# 2. Curva Fenológica
if grafico_png.exists():
    print("
🌱 Curva Temporal Média da Vegetação:")
    display(Image(filename=str(grafico_png), width=750))

# 3. Mapa Interativo (Folium)
if mapa_html.exists():
    print("
🗺️ Mapa Interativo:")
    with open(mapa_html, "r", encoding="utf-8") as f:
        html_code = f.read()
    escaped_code = html_code.replace('"', '&quot;')
    display(HTML(f'<iframe srcdoc="{escaped_code}" width="100%" height="550" style="border:none;"></iframe>'))


### 💾 Passo 7: Download dos Arquivos em .ZIP (Opcional)
Se você não utilizou o Google Drive, baixe todos os resultados (.tif, .geojson, .shp, relatórios) compactados em um único arquivo .zip:


In [ ]:
# @title 💾 Baixar Arquivos em .ZIP
import shutil
from google.colab import files

try:
    arquivo_zip = f"{slug}_resultados"
    caminho_zip = shutil.make_archive(arquivo_zip, 'zip', pasta_resultado)
    print(f"📦 Compactado com sucesso: {caminho_zip}")
    files.download(f"{arquivo_zip}.zip")
except Exception as e:
    print(f"Erro ao compactar/baixar: {e}")
